# Developed Market v7 — Data Prep with IBNR + A v E

Key changes from v6:
1. **Data-driven development**: Chain-ladder count LDFs + paid/incurred severity LDFs (replaces hardcoded BF factors for short-tail perils)
2. **A v E offset**: Computes 2021 loss ratios by line, creates `expected_claim = exposure × written_premium × LR_2021` for use as model offset
3. **Year excluded from features** (handled in modeling notebook)
4. External LDFs still applied for long-tail perils (BI, subsidence) in modeling notebook
5. All v6 cleaning retained: sum_insured fix, deduplicate claims, harmonization, age_band

In [0]:
# Step 1: Import parquet files
import pandas as pd
import numpy as np
import os

BASE = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026'
DATA_DIR = f'{BASE}/data/'
V7_DIR = f'{BASE}/files_v7/'
os.makedirs(V7_DIR, exist_ok=True)

files_to_load = [
    'policies_motor', 'policies_buildings', 'policies_contents',
    'test_motor', 'test_buildings', 'test_contents',
    'claims', 'customers', 'addresses', 'weather', 'sample_submission'
]
dfs = {}
for name in files_to_load:
    dfs[name] = pd.read_parquet(os.path.join(DATA_DIR, f'{name}.parquet'))
    print(f'{name}: {dfs[name].shape}')
print('\nAll files loaded.')

In [0]:
# Step 2: Data cleaning (same as v6 + sum_insured + deduplicate + harmonize)

# 2a. Fix exposure
for name in ['policies_motor', 'policies_buildings', 'policies_contents',
             'test_motor', 'test_buildings', 'test_contents']:
    df = dfs[name]
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    df['Year'] = ps.dt.year

# 2b. Construction type harmonization
CONSTRUCTION_MAP_DEV = {'BRICK': 'BRICK', 'STONE': 'STONE', 'W': 'TIMBER', 'T': 'TIMBER',
                        'NON_STANDARD': 'NON_STANDARD', 'TIMBER': 'TIMBER'}
for name in ['policies_buildings', 'test_buildings']:
    df = dfs[name]
    df['construction_type'] = df['construction_type'].astype(str).str.strip().str.upper().replace(CONSTRUCTION_MAP_DEV)

# 2c. Strip ID columns + zero-pad motor
id_cols = ['policy_id', 'customer_id', 'address_id', 'claim_id', 'row_id']
for name, df in dfs.items():
    for col in id_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
for name in ['policies_motor', 'test_motor']:
    dfs[name]['customer_id'] = dfs[name]['customer_id'].str.zfill(8)
dfs['customers']['customer_id'] = dfs['customers']['customer_id'].astype(str).str.zfill(8)

# 2d. Drop constant compulsory_excess
for name in ['policies_motor', 'policies_buildings', 'policies_contents']:
    if 'compulsory_excess' in dfs[name].columns and dfs[name]['compulsory_excess'].nunique() == 1:
        dfs[name].drop('compulsory_excess', axis=1, inplace=True)

# 2e. SUM_INSURED FIX (v6 thresholds)
for name in ['policies_buildings', 'policies_contents', 'test_buildings', 'test_contents']:
    df = dfs[name]
    if 'sum_insured' not in df.columns: continue
    line = 'buildings' if 'buildings' in name else 'contents'
    threshold = 10_000_000 if line == 'buildings' else 1_000_000
    flagged = df['sum_insured'] >= threshold
    if flagged.sum() > 0:
        df.loc[flagged, 'sum_insured'] = df.loc[flagged, 'sum_insured'] / 1000
        print(f'{name}: {flagged.sum()} sum_insured values fixed')

# 2f. DEDUPLICATE CLAIMS
claims = dfs['claims']
non_id_cols = [c for c in claims.columns if c != 'claim_id']
dup_mask = claims.duplicated(subset=non_id_cols, keep=False)
if dup_mask.sum() > 0:
    before = len(claims)
    claims = claims.drop_duplicates(subset=non_id_cols, keep='first').reset_index(drop=True)
    print(f'Deduplicated claims: {before} -> {len(claims)} ({dup_mask.sum()} dups removed)')
    dfs['claims'] = claims

# 2g. ADD AGE_BAND
def to_age_band(age):
    if pd.isna(age): return None
    if age <= 24: return '17-24'
    elif age <= 34: return '25-34'
    elif age <= 49: return '35-49'
    elif age <= 64: return '50-64'
    else: return '65+'
for name in ['policies_motor', 'test_motor']:
    if 'driver_age' in dfs[name].columns:
        dfs[name]['driver_age_band'] = dfs[name]['driver_age'].apply(to_age_band)
for name in ['policies_buildings', 'policies_contents', 'test_buildings', 'test_contents']:
    if 'policyholder_age' in dfs[name].columns:
        dfs[name]['policyholder_age_band'] = dfs[name]['policyholder_age'].apply(to_age_band)

print('Step 2 complete.')

In [0]:
# Step 3: Join enrichment data (same as v6)
customers_clean = dfs['customers'].drop(columns=['surname', 'date_of_birth'], errors='ignore')
addresses_clean = dfs['addresses'].copy()
weather = dfs['weather'].copy()
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)

policy_files = {'motor': 'policies_motor', 'buildings': 'policies_buildings', 'contents': 'policies_contents'}
for line, fname in policy_files.items():
    df = dfs[fname]
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(addresses_clean, on='address_id', how='left', suffixes=('', '_addr'))
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    dfs[fname] = df
    print(f'{line}: {df.shape}')

In [0]:
# Step 4: Claims processing
claims = dfs['claims']
claims['incurred_net_of_recoveries'] = claims['incurred_to_date'] - claims['recoveries']
claims['loss_year'] = pd.to_datetime(claims['loss_date']).dt.year
claims['report_date_dt'] = pd.to_datetime(claims['report_date'], errors='coerce')
claims['report_year'] = claims['report_date_dt'].dt.year
claims['dev_lag'] = claims['report_year'] - claims['loss_year']
claims['paid_incurred_ratio'] = claims['paid_to_date'] / claims['incurred_to_date'].clip(lower=0.01)
claims_active = claims[claims['status'] != 'repudiated'].copy()
print(f'Active claims: {len(claims_active)}')
print(f'Open ratio by year:')
for yr in sorted(claims_active['loss_year'].unique()):
    yr_c = claims_active[claims_active['loss_year'] == yr]
    print(f'  {yr}: {len(yr_c)} claims, {(yr_c["status"]=="open").mean():.1%} open, paid/incurred={yr_c["paid_incurred_ratio"].mean():.3f}')

In [0]:
# Step 5: DATA-DRIVEN DEVELOPMENT (replaces hardcoded BF factors)
# A) Chain-ladder on claim COUNT triangle → count LDF by dev_lag
# B) Paid/incurred ratio → severity LDF by loss_year
# C) Combined LDF = count_LDF × severity_LDF

counts = claims_active.copy()

# --- A) Chain-ladder count LDF ---
# Build cumulative count triangle
incr_triangle = counts.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
cum_triangle = incr_triangle.cumsum(axis=1)
print('=== Cumulative Claim Count Triangle ===')
print(cum_triangle.to_string())

# Compute volume-weighted age-to-age factors
dev_lags = sorted(cum_triangle.columns.tolist())
a2a_factors = {}
for i in range(len(dev_lags) - 1):
    d0, d1 = dev_lags[i], dev_lags[i + 1]
    col0 = cum_triangle[d0].values
    col1 = cum_triangle[d1].values
    mask = (col0 > 0) & (col1 > 0) & ~np.isnan(col0) & ~np.isnan(col1)
    if mask.sum() > 0:
        a2a_factors[d0] = col1[mask].sum() / col0[mask].sum()
    else:
        a2a_factors[d0] = 1.0
    print(f'  dev_lag {d0}->{d1}: factor = {a2a_factors[d0]:.4f}')

# Derive LDFs to ultimate (assume last dev_lag is ultimate = factor 1.0)
ultimate_dev = dev_lags[-1]
count_ldf_by_devlag = {ultimate_dev: 1.0}
for i in range(len(dev_lags) - 2, -1, -1):
    d = dev_lags[i]
    count_ldf_by_devlag[d] = a2a_factors.get(d, 1.0) * count_ldf_by_devlag[dev_lags[i + 1]]
print(f'\nCount LDFs to ultimate: {count_ldf_by_devlag}')

# --- B) Severity LDF from paid/incurred ratio ---
pi_by_year = claims_active.groupby('loss_year')['paid_incurred_ratio'].mean()
severity_ldf_by_year = (1 / pi_by_year).clip(lower=1.0)
print(f'\nSeverity LDFs by year:')
for yr, ldf in severity_ldf_by_year.items():
    print(f'  {yr}: paid/incurred={pi_by_year[yr]:.4f}, LDF={ldf:.4f}')

# --- C) Combined LDF ---
# Count LDF based on development AGE AT SNAPSHOT (2024), not the claim's individual report lag
# A 2021 claim reported in 2021 has dev_lag=0, but at the 2024 snapshot it's 3 years developed = mature
SNAPSHOT_YEAR = 2024
claims_active['dev_age_at_snapshot'] = SNAPSHOT_YEAR - claims_active['loss_year']
claims_active['count_ldf'] = claims_active['dev_age_at_snapshot'].map(count_ldf_by_devlag).fillna(1.0)
claims_active['severity_ldf'] = claims_active['loss_year'].map(severity_ldf_by_year).fillna(1.0)
claims_active['combined_ldf'] = claims_active['count_ldf'] * claims_active['severity_ldf']
claims_active['ultimate_claims'] = claims_active['incurred_net_of_recoveries'] * claims_active['combined_ldf']

print(f'\n=== Combined LDF by year ===')
for yr in sorted(claims_active['loss_year'].unique()):
    yr_c = claims_active[claims_active['loss_year'] == yr]
    print(f'  {yr}: count_LDF={yr_c["count_ldf"].mean():.4f}, sev_LDF={yr_c["severity_ldf"].mean():.4f}, combined={yr_c["combined_ldf"].mean():.4f}')

print(f'\nTotal ultimate: {claims_active["ultimate_claims"].sum():,.0f}')
print(f'Total incurred: {claims_active["incurred_net_of_recoveries"].sum():,.0f}')
print(f'Uplift: {(claims_active["ultimate_claims"].sum()/claims_active["incurred_net_of_recoveries"].sum()-1)*100:.1f}%')

In [0]:
# Step 5b: Match claims to policy rows and aggregate
claims_active['loss_date_dt'] = pd.to_datetime(claims_active['loss_date'])

for line, fname in policy_files.items():
    df = dfs[fname].copy()
    df['_ps_dt'] = pd.to_datetime(df['period_start'])
    df['_pe_dt'] = pd.to_datetime(df['period_end'])
    line_claims = claims_active[claims_active['line'] == line].copy()
    merged = line_claims.merge(df[['row_id', 'policy_id', '_ps_dt', '_pe_dt', 'Year']], on='policy_id', how='inner')
    merged = merged[merged['loss_date_dt'].dt.year == merged['Year']]
    mask = (merged['loss_date_dt'] >= merged['_ps_dt']) & (merged['loss_date_dt'] <= merged['_pe_dt'])
    matched = merged[mask].copy()
    matched = matched.sort_values('row_id').drop_duplicates('claim_id', keep='first')
    agg = matched.groupby('row_id').agg(
        ultimate_claims=('ultimate_claims', 'sum'),
        claim_count=('claim_id', 'count'),
        reported_incurred=('incurred_net_of_recoveries', 'sum'),
        open_claim_count=('status', lambda x: (x == 'open').sum()),
    ).reset_index()
    df = df.merge(agg, on='row_id', how='left')
    for c in ['ultimate_claims', 'reported_incurred']:
        df[c] = df[c].fillna(0)
    df['claim_count'] = df['claim_count'].fillna(0).astype(int)
    df['open_claim_count'] = df['open_claim_count'].fillna(0).astype(int)
    df.drop(['_ps_dt', '_pe_dt'], axis=1, inplace=True)
    dfs[fname] = df
    print(f'{line}: {len(df)} rows, {(df["claim_count"]>0).sum()} with claims, mean UC={df["ultimate_claims"].mean():.4f}')

In [0]:
# Step 6: A v E — compute 2021 loss ratios + create expected_claim + save proc files

# === Compute 2021 loss ratios by line ===
LR_2021 = {}
for line in ['motor', 'buildings', 'contents']:
    df = pd.read_parquet(os.path.join(DATA_DIR, f'policies_{line}.parquet'))
    df['Year'] = pd.to_datetime(df['period_start']).dt.year
    df_2021 = df[df['Year'] == 2021].copy()
    ps = pd.to_datetime(df_2021['period_start'])
    pe = pd.to_datetime(df_2021['period_end'])
    df_2021['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    ep = (df_2021['exposure'] * df_2021['written_premium']).sum()
    
    c21 = claims_active[(claims_active['loss_year'] == 2021) & (claims_active['line'] == line)]
    incurred_2021 = c21['incurred_net_of_recoveries'].sum()
    # 2021 is mature — use ultimate (which includes small dev factor)
    ultimate_2021 = c21['ultimate_claims'].sum()
    lr = ultimate_2021 / ep
    LR_2021[line] = lr
    print(f'{line}: EP={ep:,.0f}, ultimate_2021={ultimate_2021:,.0f}, LR_2021={lr:.4f}')

print(f'\nLR_2021: {LR_2021}')

# === Create expected_claim column in proc files ===
for line, fname in policy_files.items():
    df = dfs[fname]
    df['expected_claim'] = df['exposure'] * df['written_premium'] * LR_2021[line]
    df['ae_ratio'] = df['ultimate_claims'] / df['expected_claim'].clip(lower=0.01)
    print(f'{line}: expected_claim mean={df["expected_claim"].mean():.2f}, A/E mean={df["ae_ratio"].mean():.4f}')

# === Save v7 proc files ===
for line, fname in policy_files.items():
    df = dfs[fname]
    out_path = os.path.join(V7_DIR, f'proc_{line}_v7.parquet')
    df.to_parquet(out_path, index=False)
    print(f'Saved: {out_path} ({len(df)} rows, {df.shape[1]} cols)')
    # Verify sum_insured fix
    if 'sum_insured' in df.columns:
        thresh = 10_000_000 if line == 'buildings' else 1_000_000
        print(f'  sum_insured >= {thresh:,}: {(df["sum_insured"]>=thresh).sum()} (should be 0)')

# Save LR_2021 for use in modeling notebook
import json
with open(os.path.join(V7_DIR, 'lr_2021.json'), 'w') as f:
    json.dump(LR_2021, f)
print(f'\nSaved LR_2021 to {V7_DIR}lr_2021.json')
print('\n=== v7 Data Prep Complete ===')